# Digitisation tracker and sample exploration

This notebook turns the study sample (`core_sample.csv`, from `select_corpus.py`) into a working
spreadsheet for acquiring and digitising the books, and explores the sample with a few figures.

**What it does**
1. Loads the sample and adds the English edition of each book (from `harlequin_lines.csv`).
2. Builds **`digitisation_tracker.xlsx`** with one row per volume (250 books × English, French,
   German, Polish = 1,000 volumes), columns for *ordered / received / scanned / OCR checked*, an
   overview sheet that counts progress with formulas, a per-book sheet, and a reserve list.
3. Exploration: the sample by line and decade, translation lag, translators, collections, how many
   alternative editions exist per book (useful when buying second-hand).
4. Progress figures read back from the tracker.

**Re-running is safe.** If the tracker already exists, everything you have entered (dates,
vendors, prices, notes, translators from the imprint) is carried over to the new file by
`volume_id`, and the old file is kept as a dated backup.

Files needed in `DATA_DIR`: `core_sample.csv` (required); `harlequin_lines.csv`,
`usable_books.csv`, `translations_all.csv` (optional, used if present).

## 1. Setup

In [ ]:
from pathlib import Path
from datetime import datetime
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

DATA_DIR = Path(".")                              # folder with the CSV files
TRACKER = DATA_DIR / "digitisation_tracker.xlsx"
FIG_DIR = DATA_DIR / "figures_sample"
FIG_DIR.mkdir(exist_ok=True)

LANGS = ["EN", "F", "G", "P"]
LANG_NAME = {"EN": "English", "F": "French", "G": "German", "P": "Polish"}
# fixed colour per language (validated categorical palette, slots 1-4)
LANG_COLOR = {"EN": "#2a78d6", "F": "#eb6834", "G": "#1baf7a", "P": "#eda100"}
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
STEPS = ["ordered", "received", "scanned", "ocr_checked"]
STEP_LABEL = {"ordered": "Ordered", "received": "Received", "scanned": "Scanned", "ocr_checked": "OCR checked"}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "font.size": 10, "legend.frameon": False,
})

def save(fig, name):
    for ext in ("png", "svg"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")

def year_of(s):
    m = re.search(r"(1[89]\d\d|20\d\d)", str(s or ""))
    return int(m.group(1)) if m else np.nan

## 2. Load the sample

In [ ]:
sample = pd.read_csv(DATA_DIR / "core_sample.csv", dtype=str).fillna("")
print(f"{len(sample)} books in the sample")

# English edition details from FictionDB, if available
eng_cols = ["book_id", "isbn13", "isbn10", "publisher", "all_lines", "pages", "description"]
eng_path = DATA_DIR / "harlequin_lines.csv"
if eng_path.exists():
    eng = pd.read_csv(eng_path, dtype=str).fillna("")
    eng = eng[[c for c in eng_cols if c in eng.columns]].drop_duplicates("book_id")
    sample = sample.merge(eng, on="book_id", how="left").fillna("")
    print("English edition details added from harlequin_lines.csv")
else:
    print("harlequin_lines.csv not found - English ISBNs stay empty")
    for c in eng_cols[1:]:
        sample[c] = ""

sample["en_year"] = sample["pub_date"].map(year_of)
if "period" not in sample.columns:          # sample made by an older select_corpus.py
    sample["period"] = "late"
for c in [f"{L}_{k}" for L in "FGP" for k in ("id", "year", "translators", "series", "title", "isbn", "edition_type")]:
    if c not in sample.columns:
        sample[c] = ""
print(sample.groupby("period").size().rename("books").to_string())
for L in ["F", "G", "P"]:
    sample[f"{L}_year_num"] = pd.to_numeric(sample[f"{L}_year"], errors="coerce")
sample.head(3)

## 3. One row per volume

Each book becomes up to four volumes: the English original and its translations. Books before
1991 (no Polish Harlequin translations yet) are EN-FR and/or EN-DE units, books from 1991 on
EN-FR-DE-PL units (column `period` in `core_sample.csv`).
`volume_id` is the book ID for English (`6157_EN`) and the edition ID for translations (`6157_F`, `6157a_G`).

In [ ]:
rows = []
for _, b in sample.iterrows():
    rows.append({
        "volume_id": f"{b['book_id']}_EN", "book_id": b["book_id"], "language": "EN",
        "title": b["title"], "authors": b["authors"], "line": b["line"], "decade": b["decade"], "period": b["period"],
        "year": b["en_year"], "isbn": b.get("isbn13", "") or b.get("isbn10", ""),
        "series_or_collection": b.get("all_lines", "") or b["line"], "edition_type": "single",
        "translators_catalogue": "",
    })
    for L in ["F", "G", "P"]:
        if not b[f"{L}_id"]:
            continue            # e.g. no Polish edition for books before 1991 (EN-FR / EN-DE pairs)
        rows.append({
            "volume_id": b[f"{L}_id"], "book_id": b["book_id"], "language": L,
            "title": b[f"{L}_title"], "authors": b["authors"], "line": b["line"], "decade": b["decade"], "period": b["period"],
            "year": b[f"{L}_year_num"], "isbn": b[f"{L}_isbn"], "series_or_collection": b[f"{L}_series"],
            "edition_type": b[f"{L}_edition_type"], "translators_catalogue": b[f"{L}_translators"],
        })
vol = pd.DataFrame(rows)
vol["year"] = pd.to_numeric(vol["year"], errors="coerce").astype("Int64")
vol["english_title"] = vol["book_id"].map(sample.set_index("book_id")["title"])

# columns you fill in while working
WORK_COLS = ["vendor", "price", "ordered", "received", "scanned", "ocr_checked",
             "translator_imprint", "condition", "notes"]
for c in WORK_COLS:
    vol[c] = pd.Series([""] * len(vol), dtype=object)

# carry over everything already entered in an existing tracker
if TRACKER.exists():
    old = pd.read_excel(TRACKER, sheet_name="Volumes", dtype=object)      # keeps real dates as dates
    old["volume_id"] = old["volume_id"].astype(str)
    old = old.set_index("volume_id")[[c for c in WORK_COLS if c in old.columns]]
    old = old.where(old.notna(), "")
    keep = vol["volume_id"].isin(old.index)
    for c in old.columns:
        vol.loc[keep, c] = vol.loc[keep, "volume_id"].map(old[c]).values
    backup = TRACKER.with_name(f"digitisation_tracker_backup_{datetime.now():%Y%m%d_%H%M%S}.xlsx")
    TRACKER.rename(backup)
    dropped = sorted(set(old.index) - set(vol["volume_id"]))
    print(f"Carried over entries for {int(keep.sum())} volumes; old file kept as {backup.name}")
    if dropped:
        print(f"! {len(dropped)} volumes of the old tracker are no longer in the sample "
              f"(still in the backup): {dropped[:5]}{' ...' if len(dropped) > 5 else ''}")

print(vol["language"].value_counts().to_string())

## 4. Write the tracker

Sheets:
- **Volumes**: one row per volume. Enter a **date** (or any text, e.g. `x`) in *ordered*,
  *received*, *scanned*, *ocr_checked* when a step is done. `status` updates itself.
  `translator_imprint` is for the translator as printed in the book (missing in many French records).
- **Books**: one row per book with the status of all four volumes and whether the book is complete.
- **Overview**: progress per language and step, with formulas.
- **Reserve**: strict books not in the sample, by line, to replace a book that cannot be found.
- **About**: how to use the file.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.formatting.rule import FormulaRule
from openpyxl.worksheet.datavalidation import DataValidation

HEAD = Font(bold=True, color="FFFFFF")
HEAD_FILL = PatternFill("solid", fgColor="2B2B2A")
WORK_FILL = PatternFill("solid", fgColor="FFF8E5")      # columns you fill in
DONE_FILL = PatternFill("solid", fgColor="E3F4E3")
THIN = Border(bottom=Side(style="thin", color="E4E3DF"))

wb = Workbook()

# ---------- Volumes ----------
ws = wb.active
ws.title = "Volumes"
cols = ["volume_id", "book_id", "language", "english_title", "title", "authors", "line", "decade", "period",
        "year", "isbn", "series_or_collection", "edition_type", "translators_catalogue"] + WORK_COLS + ["status"]
ws.append(cols)
n = len(vol)
c_idx = {c: i + 1 for i, c in enumerate(cols)}
L = lambda c: get_column_letter(c_idx[c])
for r, (_, v) in enumerate(vol.iterrows(), start=2):
    for c in cols[:-1]:
        val = v[c]
        if not isinstance(val, str) and pd.isna(val):
            val = None
        elif c == "year":
            val = int(val)
        elif c == "price" and val != "":
            try:
                val = float(str(val).replace(",", ".").replace("€", "").replace("$", "").strip())
            except ValueError:
                pass
        elif isinstance(val, (pd.Timestamp, datetime)):
            val = pd.Timestamp(val).to_pydatetime()
        cell = ws.cell(row=r, column=c_idx[c], value=val)
        if isinstance(val, datetime):
            cell.number_format = "yyyy-mm-dd"
    o, rc, s, ocr = (f"{L(x)}{r}" for x in STEPS)
    ws.cell(row=r, column=c_idx["status"], value=(
        f'=IF({ocr}<>"","5 OCR checked",IF({s}<>"","4 scanned",IF({rc}<>"","3 received",'
        f'IF({o}<>"","2 ordered","1 to do"))))'))

widths = {"volume_id": 13, "book_id": 10, "language": 9, "english_title": 34, "title": 34, "authors": 22,
          "line": 22, "decade": 8, "period": 8, "year": 7, "isbn": 16, "series_or_collection": 24, "edition_type": 11,
          "translators_catalogue": 22, "vendor": 14, "price": 8, "ordered": 11, "received": 11,
          "scanned": 11, "ocr_checked": 12, "translator_imprint": 22, "condition": 12, "notes": 30, "status": 14}
for c, w in widths.items():
    ws.column_dimensions[L(c)].width = w
for c in WORK_COLS:
    for r in range(2, n + 2):
        ws.cell(row=r, column=c_idx[c]).fill = WORK_FILL
ws.freeze_panes = "E2"
tab = Table(displayName="Volumes", ref=f"A1:{get_column_letter(len(cols))}{n + 1}")
tab.tableStyleInfo = TableStyleInfo(name="TableStyleLight1", showRowStripes=False)
ws.add_table(tab)
ws.conditional_formatting.add(f"A2:{get_column_letter(len(cols))}{n + 1}",
                              FormulaRule(formula=[f'${L("ocr_checked")}2<>""'], fill=DONE_FILL))
dv = DataValidation(type="list", formula1='"very good,good,acceptable,poor"', allow_blank=True)
ws.add_data_validation(dv)
dv.add(f"{L('condition')}2:{L('condition')}{n + 1}")

# ---------- Books ----------
wb_books = wb.create_sheet("Books")
bcols = ["book_id", "title", "authors", "line", "decade", "EN", "F", "G", "P", "volumes_done", "volumes", "complete"]
wb_books.append(bcols)
vr = f"Volumes!${L('volume_id')}$2:${L('volume_id')}${n + 1}"
sr = f"Volumes!${L('status')}$2:${L('status')}${n + 1}"
br = f"Volumes!${L('book_id')}$2:${L('book_id')}${n + 1}"
ocr_r = f"Volumes!${L('ocr_checked')}$2:${L('ocr_checked')}${n + 1}"
vid = {(v.book_id, v.language): v.volume_id for v in vol.itertuples()}
for r, (_, b) in enumerate(sample.iterrows(), start=2):
    wb_books.append([b["book_id"], b["title"], b["authors"], b["line"], b["decade"]])
    nv = 0
    for j, lang in enumerate(LANGS):
        if (b["book_id"], lang) in vid:
            nv += 1
            wb_books.cell(row=r, column=6 + j,
                          value=f'=INDEX({sr},MATCH("{vid[(b["book_id"], lang)]}",{vr},0))')
        else:
            wb_books.cell(row=r, column=6 + j, value="-")
    wb_books.cell(row=r, column=10, value=f'=COUNTIFS({br},A{r},{ocr_r},"<>")')
    wb_books.cell(row=r, column=11, value=nv)
    wb_books.cell(row=r, column=12, value=f'=IF(J{r}=K{r},"yes","")')
for col, w in zip("ABCDEFGHIJKL", [10, 40, 24, 24, 8, 14, 14, 14, 14, 13, 9, 10]):
    wb_books.column_dimensions[col].width = w
wb_books.freeze_panes = "C2"
tb = Table(displayName="Books", ref=f"A1:L{len(sample) + 1}")
tb.tableStyleInfo = TableStyleInfo(name="TableStyleLight1", showRowStripes=False)
wb_books.add_table(tb)
wb_books.conditional_formatting.add(f"A2:L{len(sample) + 1}", FormulaRule(formula=['$L2="yes"'], fill=DONE_FILL))

# ---------- Overview ----------
ov = wb.create_sheet("Overview", 0)
ov["A1"] = "Digitisation progress"
ov["A1"].font = Font(bold=True, size=14)
ov["A2"] = (f"Sample: {len(sample)} books, {n} volumes ({int((sample['period'] == 'early').sum())} books before 1991 "
            f"as EN-FR / EN-DE units, {int((sample['period'] == 'late').sum())} from 1991 as EN-FR-DE-PL). "
            "Counts update automatically from the Volumes sheet.")
ov.append([])
ov.append(["Language", "Volumes"] + [STEP_LABEL[s] for s in STEPS] + ["% OCR checked"])
lr = f"Volumes!${L('language')}$2:${L('language')}${n + 1}"
for i, lang in enumerate(LANGS):
    r = 5 + i
    ov.cell(row=r, column=1, value=LANG_NAME[lang])
    ov.cell(row=r, column=2, value=f'=COUNTIF({lr},"{lang}")')
    for j, s in enumerate(STEPS):
        rng = f"Volumes!${L(s)}$2:${L(s)}${n + 1}"
        ov.cell(row=r, column=3 + j, value=f'=COUNTIFS({lr},"{lang}",{rng},"<>")')
    ov.cell(row=r, column=7, value=f"=IF(B{r}=0,0,F{r}/B{r})").number_format = "0%"
r = 9
ov.cell(row=r, column=1, value="All")
for col in "BCDEF":
    ov[f"{col}{r}"] = f"=SUM({col}5:{col}8)"
ov[f"G{r}"] = "=IF(B9=0,0,F9/B9)"
ov[f"G{r}"].number_format = "0%"
ov["A11"] = "Books complete (all their volumes OCR checked)"
ov["F11"] = f'=COUNTIF(Books!$L$2:$L${len(sample) + 1},"yes")'
ov["A12"] = "Money spent (sum of price)"
ov["F12"] = f"=SUM(Volumes!${L('price')}$2:${L('price')}${n + 1})"
for c in range(1, 8):
    ov.cell(row=4, column=c).font = HEAD
    ov.cell(row=4, column=c).fill = HEAD_FILL
for rr in (9, 11, 12):
    ov[f"A{rr}"].font = Font(bold=True)
ov.column_dimensions["A"].width = 44
for col in "BCDEFG":
    ov.column_dimensions[col].width = 14

# ---------- Reserve ----------
res_path = DATA_DIR / "usable_books.csv"
if res_path.exists():
    usable = pd.read_csv(res_path, dtype=str).fillna("")
    reserve = usable[(usable["level"] == "strict") & ~usable["book_id"].isin(sample["book_id"])]
    reserve = reserve[["book_id", "title", "authors", "line", "decade", "pub_date"]].sort_values(["line", "pub_date"])
    rs = wb.create_sheet("Reserve")
    rs.append(list(reserve.columns))
    for row in reserve.itertuples(index=False):
        rs.append(list(row))
    for col, w in zip("ABCDEF", [10, 40, 24, 24, 8, 12]):
        rs.column_dimensions[col].width = w
    rs.freeze_panes = "A2"
    print(f"Reserve: {len(reserve)} strict books not in the sample")

# ---------- About ----------
ab = wb.create_sheet("About")
for line in [
    "How to use this file",
    "",
    "Volumes: one row per volume (English original + French, German, Polish edition of each book).",
    "Yellow columns are yours: enter a date (or 'x') in ordered / received / scanned / ocr_checked when the step is done.",
    "status updates itself; completed volumes turn green.",
    "translator_imprint: the translator as printed on the imprint page (the catalogues miss many, especially French).",
    "Books: status of each book's volumes; '-' = no edition in that language (no Polish Harlequin before 1991); 'complete' = all its volumes OCR checked.",
    "Overview: counts per language and step, books complete, money spent.",
    "Reserve: strict books not in the sample, to replace a book that cannot be found (keep line and decade).",
    "",
    "Re-running the notebook rebuilds this file and keeps everything entered in the yellow columns.",
    f"Built {datetime.now():%Y-%m-%d %H:%M} from core_sample.csv.",
]:
    ab.append([line])
ab["A1"].font = Font(bold=True, size=13)
ab.column_dimensions["A"].width = 120

for sheet in (ws, wb_books, wb["Reserve"] if "Reserve" in wb.sheetnames else None):
    if sheet is None:
        continue
    for cell in sheet[1]:
        cell.font = HEAD
        cell.fill = HEAD_FILL
        cell.alignment = Alignment(vertical="center")

wb.save(TRACKER)
print(f"Tracker written: {TRACKER}  ({n} volumes, {len(sample)} books)")

## 5. Shopping lists per language

One CSV per language, sorted by collection and year, so you can search antiquarian sites
(ZVAB/Booklooker, Allegro, Rakuten/leboncoin, AbeBooks) collection by collection.

In [ ]:
SHOP_DIR = DATA_DIR / "shopping_lists"
SHOP_DIR.mkdir(exist_ok=True)
for lang in LANGS:
    s = (vol[vol["language"] == lang]
         [["volume_id", "title", "english_title", "authors", "series_or_collection", "year", "isbn", "edition_type"]]
         .sort_values(["series_or_collection", "year", "title"]))
    s.to_csv(SHOP_DIR / f"shopping_{LANG_NAME[lang].lower()}.csv", index=False, encoding="utf-8-sig")
    print(f"{LANG_NAME[lang]:8s} {len(s):4d} volumes, {int((s['isbn'] != '').sum()):4d} with ISBN")

## 6. Exploration

### 6.1 Composition of the sample: line × decade

In [ ]:
tab = pd.crosstab(sample["line"], sample["decade"])
tab = tab.loc[tab.sum(axis=1).sort_values(ascending=False).index]
display(tab.assign(total=tab.sum(axis=1)))

fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(tab) + 1.2))
im = ax.imshow(tab.values, cmap="Blues", aspect="auto", vmin=0)
ax.set_xticks(range(tab.shape[1]), tab.columns)
ax.set_yticks(range(tab.shape[0]), tab.index)
ax.grid(False)
for (i, j), v in np.ndenumerate(tab.values):
    if v:
        ax.text(j, i, v, ha="center", va="center", fontsize=9,
                color="white" if v > tab.values.max() * 0.55 else INK)
ax.set_title(f"Sample: {len(sample)} books by line and decade of the English edition", pad=22)
if (sample["period"] == "early").any() and "1990s" in tab.columns:
    k = list(tab.columns).index("1990s") - 0.5
    ax.axvline(k, color=INK, linewidth=1.5)
    ax.text(k - 0.08, -0.6, "EN-FR / EN-DE", ha="right", va="bottom", fontsize=8, color=INK2)
    ax.text(k + 0.08, -0.6, "EN-FR-DE-PL", ha="left", va="bottom", fontsize=8, color=INK2)
for s in ax.spines.values():
    s.set_visible(False)
save(fig, "01_line_decade")
plt.show()

### 6.2 Translation lag: years between the English edition and the chosen translation

In [ ]:
lag = pd.DataFrame({LANG_NAME[L]: sample[f"{L}_year_num"] - sample["en_year"] for L in ["F", "G", "P"]})
display(lag.describe().round(1).T[["count", "mean", "25%", "50%", "75%", "max"]])

fig, ax = plt.subplots(figsize=(7.5, 3.2))
langs = ["F", "G", "P"]
data = [lag[LANG_NAME[L]].dropna() for L in langs]
bp = ax.boxplot(data, vert=False, widths=0.5, patch_artist=True, showfliers=True,
                medianprops=dict(color=INK, linewidth=2),
                flierprops=dict(marker="o", markersize=4, markerfacecolor=INK2, markeredgecolor="none", alpha=0.5))
for patch, L in zip(bp["boxes"], langs):
    patch.set_facecolor(LANG_COLOR[L]); patch.set_edgecolor(SURFACE); patch.set_alpha(0.9)
for el in ("whiskers", "caps"):
    for line in bp[el]:
        line.set_color(INK2)
ax.set_yticks(range(1, 4), [LANG_NAME[L] for L in langs])
ax.axvline(0, color=INK2, linewidth=1)
ax.set_xlabel("years after the English edition (chosen edition)")
med = {LANG_NAME[L]: lag[LANG_NAME[L]].median() for L in langs}
ax.set_title("Median translation lag: " + ", ".join(f"{k} {v:.0f}" for k, v in med.items()) + " years")
ax.grid(axis="y", visible=False)
save(fig, "02_translation_lag")
plt.show()
print("Negative values: the translation appeared before the North American edition - usually a UK "
      "Mills & Boon first edition that the translation was made from.")

### 6.3 Publication years: English original vs. translation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharex=True, sharey=True)
lo = int(np.nanmin(sample["en_year"])) - 2
hi = int(np.nanmax(sample[[f"{L}_year_num" for L in "FGP"]].max())) + 2
for ax, L in zip(axes, ["F", "G", "P"]):
    ax.plot([lo, hi], [lo, hi], color=INK2, linewidth=1, linestyle="--")
    ax.scatter(sample["en_year"], sample[f"{L}_year_num"], s=22, color=LANG_COLOR[L],
               edgecolor=SURFACE, linewidth=0.8, alpha=0.85)
    ax.set_title(LANG_NAME[L], fontsize=11)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel("English edition")
axes[0].set_ylabel("chosen translation")
fig.suptitle("Points above the dashed line were translated later; far above = a later reissue was chosen",
             x=0.01, ha="left", fontsize=12, fontweight="bold", color=INK)
fig.tight_layout()
save(fig, "03_years_scatter")
plt.show()

### 6.4 Translators: known from the catalogue vs. to be taken from the imprint page

In [ ]:
has = {L: sample[f"{L}_id"] != "" for L in "FGP"}
tr = pd.DataFrame({
    "known": [int(((sample[f"{L}_translators"] != "") & has[L]).sum()) for L in "FGP"],
    "from imprint": [int(((sample[f"{L}_translators"] == "") & has[L]).sum()) for L in "FGP"],
}, index=[LANG_NAME[L] for L in "FGP"])
display(tr.assign(share_known=(tr["known"] / tr.sum(axis=1)).map("{:.0%}".format)))

fig, ax = plt.subplots(figsize=(7.5, 2.6))
y = np.arange(len(tr))
ax.barh(y, tr["known"], color=[LANG_COLOR[L] for L in "FGP"], height=0.55, edgecolor=SURFACE, linewidth=2)
ax.barh(y, tr["from imprint"], left=tr["known"], color="#d8d7d2", height=0.55, edgecolor=SURFACE, linewidth=2)
for i, (k, m) in enumerate(zip(tr["known"], tr["from imprint"])):
    ax.text(k / 2, i, f"{k} known", ha="center", va="center", color="white" if k > 25 else INK, fontsize=9)
    if m:
        ax.text(k + m / 2, i, f"{m} from imprint", ha="center", va="center", color=INK, fontsize=9)
ax.set_yticks(y, tr.index)
ax.invert_yaxis()
ax.set_xlabel("editions in the sample")
share = tr["known"] / tr.sum(axis=1)
ax.set_title(f"Translator names: catalogues name {share.min():.0%} for {share.idxmin()}, "
             f"{share.max():.0%} for {share.idxmax()}")
ax.grid(axis="y", visible=False)
save(fig, "04_translators")
plt.show()

top = {}
for L in "FGP":
    names = sample[f"{L}_translators"].str.split(";").explode().str.strip()
    top[LANG_NAME[L]] = names[names != ""].value_counts().head(8)
for k, v in top.items():
    print(f"\n{k}: most frequent translators in the sample\n{v.to_string()}")

### 6.5 Collections the chosen editions come from

In [ ]:
def collection(s):
    s = str(s).split("=")[0]                                          # "Baccara Band 2294 = 23"
    s = re.sub(r"\[[^\]]*\]", "", s)                                  # "[...]"
    s = re.sub(r"\s*[:;,]\s*(?=[:;,])", "", s)                          # leftover ": ,"
    s = re.sub(r"[\s.,;:#-]*\d+[\s\w/()-]*$", "", s)                    # trailing number
    s = re.sub(r"[\s.,;:]*\b(nr|no|n°|bd|band|vol|t)\.?\s*$", "", s, flags=re.I)  # "; nr", "Bd"
    s = re.sub(r"\s*[:;,.]\s*$", "", s)
    s = re.sub(r"\s+", " ", s).strip(" .,;:")
    return s or "(none)"

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, L in zip(axes, "FGP"):
    c = sample[f"{L}_series"].map(collection).value_counts().head(8)[::-1]
    ax.barh(c.index, c.values, color=LANG_COLOR[L], height=0.6)
    for i, v in enumerate(c.values):
        ax.text(v, i, f" {v}", va="center", fontsize=9, color=INK2)
    ax.set_title(LANG_NAME[L], fontsize=11)
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", labelsize=8)
fig.suptitle("Top collections per language (what to search for when buying)",
             x=0.01, ha="left", fontsize=12, fontweight="bold", color=INK)
fig.tight_layout()
save(fig, "05_collections")
plt.show()

### 6.6 How many editions exist per book and language?

More editions (reprints, anthologies, re-issues) mean a better chance of finding a copy
second-hand. Books with a single edition in some language are the ones to order first.

In [ ]:
tpath = DATA_DIR / "translations_all.csv"
if tpath.exists():
    t = pd.read_csv(tpath, dtype=str).fillna("")
    t = t[t["book_id"].isin(sample["book_id"])]
    if "source" in t.columns:
        t = t[~t["source"].str.contains("name match", case=False)]     # e-book shop products
    ed = t.groupby(["book_id", "language_letter"]).size().unstack().reindex(columns=list("FGP")).fillna(0).astype(int)
    for L in "FGP":                                   # count only the languages in each book's unit
        ed.loc[~ed.index.isin(sample.loc[sample[f"{L}_id"] != "", "book_id"]), L] = 0
    display(ed.replace(0, np.nan).describe().round(1).T[["count", "mean", "50%", "max"]].rename(index=LANG_NAME))

    fig, ax = plt.subplots(figsize=(7.5, 3))
    bins = np.arange(1, 8) - 0.5
    width = 0.27
    for k, L in enumerate("FGP"):
        counts = ed.loc[ed[L] > 0, L].clip(upper=6).value_counts().reindex(range(1, 7), fill_value=0)
        ax.bar(counts.index + (k - 1) * width, counts.values, width=width * 0.92,
               color=LANG_COLOR[L], label=LANG_NAME[L])
    ax.set_xticks(range(1, 7), ["1", "2", "3", "4", "5", "6+"])
    ax.set_xlabel("editions of the book in that language")
    ax.set_ylabel("books")
    ax.legend(ncol=3, loc="upper right")
    ax.set_title("Most books exist in a single edition per language")
    ax.grid(axis="x", visible=False)
    save(fig, "06_editions_per_book")
    plt.show()

    scarce = ed[((ed == 1) | (ed == 0)).all(axis=1) & (ed == 1).any(axis=1)].index
    print(f"{len(scarce)} books have exactly one edition in every language of their unit -> order these first.")
    (sample[sample["book_id"].isin(scarce)][["book_id", "title", "authors", "line", "pub_date"]]
     .to_csv(DATA_DIR / "order_first.csv", index=False, encoding="utf-8-sig"))
else:
    print("translations_all.csv not found - section skipped")

## 7. Progress (read back from the tracker)

Run this section any time after updating the spreadsheet. It reads the yellow columns
directly, so it works even if the formulas have not been recalculated.

In [ ]:
cur = pd.read_excel(TRACKER, sheet_name="Volumes", dtype=str).fillna("")
prog = pd.DataFrame({STEP_LABEL[s]: cur.groupby("language")[s].apply(lambda x: (x.str.strip() != "").sum())
                     for s in STEPS}).reindex(LANGS)
prog.index = [LANG_NAME[L] for L in LANGS]
total = cur.groupby("language").size().reindex(LANGS).values
display(prog.assign(volumes=total))

# furthest step reached per volume -> stacked bar per language
def stage(r):
    for s in reversed(STEPS):
        if str(r[s]).strip():
            return STEP_LABEL[s]
    return "To do"
cur["stage"] = cur.apply(stage, axis=1)
order = ["OCR checked", "Scanned", "Received", "Ordered", "To do"]
shade = {"OCR checked": "#1c5cab", "Scanned": "#3987e5", "Received": "#6da7ec", "Ordered": "#b7d3f6", "To do": "#ecebe7"}
st = pd.crosstab(cur["language"], cur["stage"]).reindex(index=LANGS, columns=order, fill_value=0)

fig, ax = plt.subplots(figsize=(8, 2.8))
left = np.zeros(len(st))
for s in order:
    ax.barh(range(len(st)), st[s], left=left, color=shade[s], height=0.6, edgecolor=SURFACE, linewidth=2, label=s)
    left += st[s].values
ax.set_yticks(range(len(st)), [LANG_NAME[L] for L in LANGS])
ax.invert_yaxis()
ax.set_xlabel("volumes")
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.32), fontsize=9)
done = int((cur["stage"] == "OCR checked").sum())
ax.set_title(f"Digitisation progress: {done} of {len(cur)} volumes OCR checked")
ax.grid(axis="y", visible=False)
save(fig, "07_progress")
plt.show()

books_done = cur[cur["stage"] == "OCR checked"].groupby("book_id").size()
need = cur.groupby("book_id").size()
print(f"Books complete (all their volumes OCR checked): {int((books_done.reindex(need.index).fillna(0) == need).sum())} "
      f"of {cur['book_id'].nunique()}")

## 8. Progress over time

Uses the dates entered in the tracker (cells that are not dates, such as `x`, are ignored here).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
plotted = False
for s in STEPS:
    d = pd.to_datetime(cur[s], errors="coerce").dropna().sort_values()
    if len(d):
        ax.step(d, np.arange(1, len(d) + 1), where="post", linewidth=2,
                color={"ordered": "#b7d3f6", "received": "#6da7ec", "scanned": "#3987e5", "ocr_checked": "#1c5cab"}[s],
                label=STEP_LABEL[s])
        plotted = True
if plotted:
    ax.axhline(len(cur), color=INK2, linewidth=1, linestyle="--")
    ax.text(ax.get_xlim()[0], len(cur), f" target {len(cur)}", va="bottom", color=INK2, fontsize=9)
    ax.set_ylabel("volumes (cumulative)")
    loc = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    ax.set_ylim(0, len(cur) * 1.08)
    ax.legend(loc="center right")
    ax.set_title("Cumulative volumes per step")
    save(fig, "08_progress_over_time")
    plt.show()
else:
    plt.close(fig)
    print("No dates entered yet.")